# OrbiFragsNets annotation of ms2Topo spectra

Start after ms2Topo consensus formation. This workflow requires numpy, pandas,
and scipy, plus your Jupyter environment. No raw mzML or pyOpenMS is needed.
Run from the OrbiFragsNets repository directory.

Consensus IQR is converted to a configurable uncertainty proxy. It is not a
raw-profile confidence interval. The existing singly charged positive-ion
mass convention and valence constraints remain. No adduct conversion occurs.
Network enumeration is bounded by max_networks to keep large searches explicit.


In [ ]:
from pathlib import Path
from Functions.ms2topo import annotate_ms2topo_features, annotate_ms2topo_consensus_spectrum

example = Path("examples/ms2topo")
features_table = example / "ms2Topo_Features_table-0.csv"
consensus_folder = example / "Alignedms2Features2026-05-20"


## Annotate selected features

Replace the example paths with your outputs. The bundled subset contains
consensus files for features 0 and 1 only. Precursor masses come from the
feature table. The strongest 10 usable fragments are used by default;
set top_n=None to keep all, subject to the search limit.


In [ ]:
result = annotate_ms2topo_features(
    features_table = features_table,
    consensus_spectra_folder = consensus_folder,
    feat_ids = [0, 1],
    precursor_ci_ppm = 10.0,
    min_ci_ppm = 10.0,
    max_ci_ppm = 30.0,
    top_n = 10,
    max_networks = 100000,
)
result["diagnostics"]


In [ ]:
result["annotations"]


## Inspect the converted spectrum

Source columns and row indices are retained. The default median_Int removes
zero-median fragments; intensity_col="mean_Int" explicitly selects means.
Selected intensities sum to 100 after product-ion filtering and peak selection.


In [ ]:
result["spectrum_peaks"]["0"]


## Export results

Review all diagnostic statuses. Expected no-annotation cases have empty
annotations and explicit statuses. Invalid inputs and engine errors raise by
default; batch on_error="record" records each failure and continues.


In [ ]:
output = Path("annotation_results")
output.mkdir(exist_ok=True)
result["annotations"].to_csv(output / "annotations.csv", index=False)
result["diagnostics"].to_csv(output / "diagnostics.csv", index=False)


## Single-spectrum alternative

Precursor m/z must be supplied; it is never inferred from the largest fragment.


In [ ]:
single = annotate_ms2topo_consensus_spectrum(
    consensus_df = consensus_folder / "Consensus_ms2-spectra_0.csv",
    precursor_mz = 245.0781752822625,
    feat_id = "0",
    top_n = 10,
)
single["annotations"]
